In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
import torchbnn as bnn
from tqdm.auto import tqdm
import pyro
import pyro.distributions as dist
from torch.distributions import constraints
from pyro.infer import SVI, Trace_ELBO
from pyro.optim import ClippedAdam
import math




# Установка случайного сида для воспроизводимости
torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"

/home/nesaulov/study/urfu/coursach/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('../data/data.csv')
df

,"Sigma, Mpa",T.K,t.h,Th,C,Cr,Co,Mo,W,Al,...,La,S,Si,Mn,P,Hf,Cu,Ge,Ga,Ni
0,241.316495,1144.2600,4.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,NaN,99.500
1,241.316495,1144.2600,113.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.75,NaN,NaN,NaN,99.250
2,241.316495,1144.2600,68.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,0.05,99.450
3,241.316495,1144.2600,40.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,0.20,99.300
4,241.316495,1144.2600,32.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,0.50,99.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3441,1061.792578,922.0389,1183.7,NaN,0.02,17.0,28.4,3.4,1.9,1.03,...,NaN,6.0,0.05,0.05,NaN,NaN,0.02,NaN,NaN,33.904
3442,1103.161120,866.4833,25554.7,NaN,0.02,17.0,28.4,3.4,1.9,1.03,...,NaN,6.0,0.05,0.05,NaN,NaN,0.02,NaN,NaN,33.904
3443,241.316495,1199.8170,183.2,NaN,0.18,10.0,15.0,3.0,NaN,5.50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,59.750
3444,241.316495,1199.8170,153.0,NaN,0.18,10.0,15.0,3.0,NaN,5.50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,59.750


In [3]:
df.duplicated().sum()


np.int64(171)

In [4]:
df = df.drop_duplicates()


In [5]:
df

,"Sigma, Mpa",T.K,t.h,Th,C,Cr,Co,Mo,W,Al,...,La,S,Si,Mn,P,Hf,Cu,Ge,Ga,Ni
0,241.316495,1144.2600,4.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,NaN,99.500
1,241.316495,1144.2600,113.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.75,NaN,NaN,NaN,99.250
2,241.316495,1144.2600,68.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,0.05,99.450
3,241.316495,1144.2600,40.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,0.20,99.300
4,241.316495,1144.2600,32.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,0.50,99.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3441,1061.792578,922.0389,1183.7,NaN,0.02,17.0,28.4,3.4,1.9,1.03,...,NaN,6.0,0.05,0.05,NaN,NaN,0.02,NaN,NaN,33.904
3442,1103.161120,866.4833,25554.7,NaN,0.02,17.0,28.4,3.4,1.9,1.03,...,NaN,6.0,0.05,0.05,NaN,NaN,0.02,NaN,NaN,33.904
3443,241.316495,1199.8170,183.2,NaN,0.18,10.0,15.0,3.0,NaN,5.50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,59.750
3444,241.316495,1199.8170,153.0,NaN,0.18,10.0,15.0,3.0,NaN,5.50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,59.750


In [ ]:
df_diff = df.copy()


0       False
1       False
2       False
3       False
4       False
        ...  
3441     True
3442     True
3443    False
3444     True
3445     True
Length: 3275, dtype: bool

In [84]:
col = "Sigma, Mpa"
other_cols = df.columns.drop(col)

result = (
    df.groupby(list(other_cols), dropna=False)[col]
      .agg(lambda x: list(x))
      .reset_index(name=f"{col}_values")
)

result


,T.K,t.h,Th,C,Cr,Co,Mo,W,Al,Ti,...,S,Si,Mn,P,Hf,Cu,Ge,Ga,Ni,"Sigma, Mpa_values"
0,294.260,120.0,NaN,0.10,9.0,20.0,1.0,5.5,5.50,NaN,...,NaN,0.01,NaN,NaN,0.20,NaN,NaN,NaN,51.030,[1186.5876797]
1,773.150,100.0,0.0,0.00,5.0,0.0,2.0,5.0,5.50,1.0,...,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.0,69.500,[880.0]
2,773.150,1000.0,0.0,0.00,10.0,5.0,1.5,7.0,5.00,1.4,...,0.0,0.00,0.0,0.0,0.10,0.0,0.0,0.0,62.000,[154.0]
3,823.150,100.0,0.0,0.15,9.0,10.0,0.0,10.0,5.50,1.5,...,0.0,0.00,0.0,0.0,1.20,0.0,0.0,0.0,58.085,[880.0]
4,866.480,4.0,0.0,NaN,19.0,0.0,3.0,NaN,0.50,0.9,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,53.000,[1151.424419]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1044,1473.150,65.0,NaN,NaN,4.2,8.0,1.0,4.0,5.40,1.0,...,NaN,0.10,NaN,NaN,0.25,NaN,NaN,NaN,62.650,[80.0]
1045,1473.150,100.0,0.0,0.06,16.0,5.0,3.0,0.0,3.50,3.5,...,0.0,0.00,0.0,0.0,1.00,0.0,0.0,0.0,62.910,"[5.0, 15.6, 22.9, 23.0, 23.5, 27.0, 30.0, 31.0..."
1046,1473.150,500.0,0.0,0.03,2.0,16.5,2.0,6.0,5.55,0.0,...,0.0,0.00,0.0,0.0,0.15,0.0,0.0,0.0,48.570,"[10.0, 24.0, 25.0, 28.0, 30.0, 34.7, 37.0, 38.0]"
1047,1560.928,90.0,NaN,NaN,10.0,5.0,NaN,4.0,5.00,1.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62.500,[248.211252]


In [85]:
df2 = result
df2["Sigma, Mpa"] = df2["Sigma, Mpa_values"].apply(np.mean)
df2 = df2.drop(columns=["Sigma, Mpa_values"])
df2

,T.K,t.h,Th,C,Cr,Co,Mo,W,Al,Ti,...,S,Si,Mn,P,Hf,Cu,Ge,Ga,Ni,"Sigma, Mpa"
0,294.260,120.0,NaN,0.10,9.0,20.0,1.0,5.5,5.50,NaN,...,NaN,0.01,NaN,NaN,0.20,NaN,NaN,NaN,51.030,1186.587680
1,773.150,100.0,0.0,0.00,5.0,0.0,2.0,5.0,5.50,1.0,...,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.0,69.500,880.000000
2,773.150,1000.0,0.0,0.00,10.0,5.0,1.5,7.0,5.00,1.4,...,0.0,0.00,0.0,0.0,0.10,0.0,0.0,0.0,62.000,154.000000
3,823.150,100.0,0.0,0.15,9.0,10.0,0.0,10.0,5.50,1.5,...,0.0,0.00,0.0,0.0,1.20,0.0,0.0,0.0,58.085,880.000000
4,866.480,4.0,0.0,NaN,19.0,0.0,3.0,NaN,0.50,0.9,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,53.000,1151.424419
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1044,1473.150,65.0,NaN,NaN,4.2,8.0,1.0,4.0,5.40,1.0,...,NaN,0.10,NaN,NaN,0.25,NaN,NaN,NaN,62.650,80.000000
1045,1473.150,100.0,0.0,0.06,16.0,5.0,3.0,0.0,3.50,3.5,...,0.0,0.00,0.0,0.0,1.00,0.0,0.0,0.0,62.910,57.678889
1046,1473.150,500.0,0.0,0.03,2.0,16.5,2.0,6.0,5.55,0.0,...,0.0,0.00,0.0,0.0,0.15,0.0,0.0,0.0,48.570,28.337500
1047,1560.928,90.0,NaN,NaN,10.0,5.0,NaN,4.0,5.00,1.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62.500,248.211252


In [86]:
# df2.to_csv('../data/data_clear.csv', index=False)

In [87]:
df3 = df.copy()
df3 = df3.drop(columns=["Sigma, Mpa", "t.h"])
df3 = df3.drop_duplicates().reset_index().drop(columns='index')
df3


,T.K,Th,C,Cr,Co,Mo,W,Al,Ti,Nb,...,La,S,Si,Mn,P,Hf,Cu,Ge,Ga,Ni
0,1144.2600,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,NaN,99.500
1,1144.2600,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.75,NaN,NaN,NaN,99.250
2,1144.2600,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,0.05,99.450
3,1144.2600,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,0.20,99.300
4,1144.2600,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,0.50,99.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
844,1005.3720,NaN,0.02,17.0,28.4,3.4,1.9,1.03,3.1,1.1,...,NaN,6.0,0.05,0.05,NaN,0.00,0.02,NaN,NaN,33.904
845,1005.3720,NaN,0.02,17.0,28.4,3.4,1.9,1.03,3.1,1.1,...,NaN,6.0,0.05,0.05,NaN,NaN,0.02,NaN,NaN,33.904
846,922.0389,NaN,0.02,17.0,28.4,3.4,1.9,1.03,3.1,1.1,...,NaN,6.0,0.05,0.05,NaN,NaN,0.02,NaN,NaN,33.904
847,866.4833,NaN,0.02,17.0,28.4,3.4,1.9,1.03,3.1,1.1,...,NaN,6.0,0.05,0.05,NaN,NaN,0.02,NaN,NaN,33.904


In [88]:
col = "T.K"
other_cols = df3.columns.drop(col)

result3 = (
    df3.groupby(list(other_cols), dropna=False)[col]
      .agg(lambda x: list(x))
      .reset_index(name=f"{col}_values")
)

result3

,Th,C,Cr,Co,Mo,W,Al,Ti,Nb,B,...,S,Si,Mn,P,Hf,Cu,Ge,Ga,Ni,T.K_values
0,0.0,0.0,2.0,3.0,0.4,5.00,5.7,0.20,0.1,0.0,...,0.0,0.0,0.0,0.0,0.030,0.0,0.0,0.0,67.570,[1255.15]
1,0.0,0.0,2.0,3.0,0.4,5.00,5.7,0.20,0.1,0.0,...,0.0,0.0,0.0,0.0,0.150,0.0,0.0,0.0,67.450,[1173.15]
2,0.0,0.0,5.0,0.0,2.0,5.00,5.5,1.00,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,69.500,[773.15]
3,0.0,0.0,5.2,7.5,3.0,8.75,5.3,0.55,0.0,0.0,...,0.0,0.0,0.0,0.0,0.105,0.0,0.0,0.0,60.345,[1073.15]
4,0.0,0.0,6.0,9.0,0.6,6.00,5.6,1.00,0.0,0.0,...,0.0,0.0,0.0,0.0,0.100,0.0,0.0,0.0,59.700,[1273.15]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
487,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.500,NaN,NaN,0.2,99.300,[1144.26]
488,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.500,NaN,NaN,0.5,99.000,[1144.26]
489,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.500,NaN,NaN,NaN,99.500,[1144.26]
490,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.750,NaN,NaN,0.2,99.050,[1144.26]


In [89]:
df3 = result3
df3["T.K"] = df3["T.K_values"].apply(np.median)
df3 = df3.drop(columns=["T.K_values"])
df3

,Th,C,Cr,Co,Mo,W,Al,Ti,Nb,B,...,S,Si,Mn,P,Hf,Cu,Ge,Ga,Ni,T.K
0,0.0,0.0,2.0,3.0,0.4,5.00,5.7,0.20,0.1,0.0,...,0.0,0.0,0.0,0.0,0.030,0.0,0.0,0.0,67.570,1255.15
1,0.0,0.0,2.0,3.0,0.4,5.00,5.7,0.20,0.1,0.0,...,0.0,0.0,0.0,0.0,0.150,0.0,0.0,0.0,67.450,1173.15
2,0.0,0.0,5.0,0.0,2.0,5.00,5.5,1.00,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,69.500,773.15
3,0.0,0.0,5.2,7.5,3.0,8.75,5.3,0.55,0.0,0.0,...,0.0,0.0,0.0,0.0,0.105,0.0,0.0,0.0,60.345,1073.15
4,0.0,0.0,6.0,9.0,0.6,6.00,5.6,1.00,0.0,0.0,...,0.0,0.0,0.0,0.0,0.100,0.0,0.0,0.0,59.700,1273.15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
487,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.500,NaN,NaN,0.2,99.300,1144.26
488,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.500,NaN,NaN,0.5,99.000,1144.26
489,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.500,NaN,NaN,NaN,99.500,1144.26
490,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.750,NaN,NaN,0.2,99.050,1144.26


In [90]:
# df3.to_csv("../data/data_clear_2.csv", index=False)

In [ ]:
# df3 = pd.read_csv("../data/data_clear_2.csv")

In [96]:
dups = df3[df3.duplicated(subset=df3.columns.drop("T.K"), keep=False)]
dups = dups.sort_values(list(df3.columns))
print(dups)


Empty DataFrame
Columns: [Th, C, Cr, Co, Mo, W, Al, Ti, Nb, B, Fe, Y, Zr, Ta, Re, Ru, V, La, S, Si, Mn, P, Hf, Cu, Ge, Ga, Ni, T.K]
Index: []

[0 rows x 28 columns]
